## Gold Preprocessing (Deliverable 1.3.1)

This notebook completes the Gold-layer preprocessing stage of the Medallion Architecture. It prepares the pump sensor dataset for model training and for the comparative evaluation described in Section C of the project proposal.

**Purpose:**  
To transform the cleaned Silver-layer dataset into a fully model-ready Gold dataset using the final feature registry, imputation decisions, and anomaly labels produced during Silver EDA. This ensures that both the baseline model and the three-stage cascade model are trained on a consistent and reproducible feature set.

**Key Goals:**

- Load the finalized Silver dataset and feature registry.
- Apply the Silver EDA imputation strategy (forward-fill followed by median).
- Standardize and scale feature values as required for model stability.
- Construct the model-ready Gold dataframe with:
  - 50 vetted numeric features for Stage 1 (broad) modeling,
  - A reduced feature subset for Stage 2 (narrow) modeling,
  - Profile- and rule-based sensor groupings for Stage 3 confirmation logic.
- Generate and export all Gold-layer preprocessing artifacts, including:
  - The Gold preprocessed parquet dataset,
  - Stage 1 and Stage 2 feature sets,
  - Stage 3 rule/profile sensor lists,
  - A preprocessing summary and metadata record.

**Relevance to Section C:**  
Outputs from this notebook directly support the methods described in Section C by:

- Providing a stable, aligned feature matrix for the baseline Isolation Forest and the three-stage cascade (C.2, C.2.A).
- Ensuring consistent preprocessing necessary for the paired model comparison and alert-quality evaluation (C.4).
- Supplying the structured Gold dataset that underpins the visual communication of alert patterns and model differences (C.6).

This notebook finalizes the dataset that the Gold Modeling notebook will use to implement, evaluate, and compare both anomaly-detection approaches.

## Gold Preprocessing Setup and Imports

### Ask

Why am I loading these utilities before preprocessing?

### Answer

Gold preprocessing depends on configuration loading, saved Silver artifacts, truth-record handling, feature utilities, scaling, imputation, and artifact writing.

This cell prepares the notebook environment so the remaining steps can turn the cleaned Silver output into model-ready Gold datasets.

In [26]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, cast


from pathlib import Path
import yaml
import os
import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


# Custom Utilities Module
from utils.core.paths import get_paths

from utils.core.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json, 
)

from utils.core.logging_profiler import profile_dataframe

from utils.core.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.core.wandb_utils import finalize_wandb_stage

from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.core.config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.medallion.gold.cascade_row_tracking import ensure_stable_row_id

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)

from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)

# Ledger 
from utils.core.ledger import Ledger

# Artifacts
from utils.core.artifacts import (
    build_artifact_dirs_from_config,
    artifact_file_path,
)

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

In [27]:
def require_dict(value: Any | None, name: str) -> Dict[str, Any]:
    if value is None:
        return {}

    if not isinstance(value, dict):
        raise TypeError(
            f"{name} must be a dictionary, got {type(value).__name__}: {value!r}"
        )

    return cast(Dict[str, Any], value)


def require_list(value: Any | None, name: str) -> List[Any]:
    if value is None:
        return []

    if not isinstance(value, list):
        raise TypeError(
            f"{name} must be a list, got {type(value).__name__}: {value!r}"
        )

    return cast(List[Any], value)


def normalize_feature_columns(value: Any, name: str = "feature_columns") -> list[str]:
    """
    Normalize feature-column results into list[str].

    Handles:
    - list[str]
    - tuple[list[str], dict]
    - single string
    """
    if isinstance(value, tuple):
        if len(value) == 0:
            return []
        value = value[0]

    if isinstance(value, str):
        raw_values = [value]
    elif isinstance(value, list):
        raw_values = value
    else:
        raise TypeError(
            f"{name} must be a list[str], tuple[list[str], dict], or str; "
            f"got {type(value).__name__}: {value!r}"
        )

    return [
        str(column_name).strip()
        for column_name in raw_values
        if str(column_name).strip() != ""
    ]

---

## Load Paths, Configuration, and Runtime Settings

In this section I am loading the resolved project paths and the Gold preprocessing configuration that controls how this notebook runs.

The main purpose here is to centralize the values that drive the workflow before I start transforming the dataframe. That includes:
- project folders
- Gold version and recipe information
- pipeline and run mode settings
- dataset identity
- artifact locations
- truth-store settings
- experiment tracking settings

I like doing this near the top because it keeps the rest of the notebook cleaner and more repeatable. Instead of scattering hard-coded values throughout the workflow, I define them once here and reuse them downstream.

In [28]:

paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# Get configs
#CONFIG_ROOT = Path("configs")
CONFIG_ROOT = paths.configs

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# Temporary selector until you centralize mode selection
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="gold_preprocessing",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

GOLD_CFG = CONFIG["gold_preprocessing"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# ---- Stage details ----
STAGE = "gold"
LAYER_NAME = GOLD_CFG["layer_name"]
GOLD_VERSION = CONFIG["versions"]["gold"]
RECIPE_ID = GOLD_CFG["recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

GOLD_PROCESS_RUN_ID = make_process_run_id(GOLD_CFG["process_run_id_prefix"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  
# ---- W&B ----
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{GOLD_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# ---- File names ----
SILVER_FILE_NAME = FILENAMES["silver_train_file_name"]


GOLD_FILE_NAME = FILENAMES["gold_preprocessed_file_name"]
GOLD_TRAIN_FILE_NAME = FILENAMES["gold_train_file_name"]
GOLD_TEST_FILE_NAME = FILENAMES["gold_test_file_name"]
GOLD_FIT_FILE_NAME = FILENAMES["gold_fit_file_name"]

GOLD_PRESCALED_FILE_NAME = FILENAMES["gold_preprocessed_prescaled_file_name"]
GOLD_SCALED_FILE_NAME = FILENAMES["gold_preprocessed_scaled_file_name"]

FEATURE_REGISTRY_FILE_NAME = FILENAMES["feature_registry_file_name"]
IMPUTE_RECOMMENDATION_FILE_NAME = FILENAMES["impute_recommendation_file_name"]

STAGE1_FEATURES_FILE_NAME = FILENAMES["stage1_features_file_name"]
STAGE2_FEATURES_FILE_NAME = FILENAMES["stage2_features_file_name"]
STAGE3_PRIMARY_FILE_NAME = FILENAMES["stage3_primary_file_name"]
STAGE3_SECONDARY_FILE_NAME = FILENAMES["stage3_secondary_file_name"]

CASCADE_DEFAULTS_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_defaults_results_file_name_csv"]
CASCADE_DEFAULTS_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_defaults_results_file_name_pickle"]

CASCADE_TUNED_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_tuned_results_file_name_csv"]
CASCADE_TUNED_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_tuned_results_file_name_pickle"]

COMPARISON_FILE_NAME = FILENAMES["comparison_file_name"]

GOLD_PREPROCESSING_LEDGER_FILE_NAME = FILENAMES["gold_preprocessing_ledger_file_name"]

PREPROCESSING_SUMMARY_FILE_NAME = FILENAMES["preprocessing_summary_file_name"]
PREPROCESSING_METADATA_FILE_NAME = FILENAMES["preprocessing_metadata_file_name"]
REFERENCE_PROFILE_FILE_NAME = FILENAMES["reference_profile_file_name"]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# ---- Runtime knobs ----
TRAIN_FRACTION = float(GOLD_CFG["train_fraction"])
RANDOM_SEED = int(GOLD_CFG["random_seed"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

SCALER_KIND = GOLD_CFG["scaler_kind"]
SCALER_ARTIFACT_NAME_TEMPLATE = GOLD_CFG["scaler_artifact_name_template"]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

STAGE2_TARGET_FEATURE_COUNT = int(GOLD_CFG["stage2_target_feature_count"])
STAGE3_PRIMARY_COUNT = int(GOLD_CFG["stage3_primary_count"])
STAGE3_SECONDARY_COUNT = int(GOLD_CFG["stage3_secondary_count"])


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  
# ---- Paths setup ----
SILVER_DATA_PATH = Path(PATHS["data_silver_train_dir"])
SILVER_TRAIN_DATA_PATH = Path(PATHS["silver_train_data_path"])
SILVER_PROFILED_DATAFRAME_DATA_PATH = Path(PATHS["silver_subsets_profiled_dataframe_data_path"])
SILVER_NORMAL_CLEAN_DATA_PATH = Path(PATHS["silver_subsets_normal_clean_data_path"])
SILVER_NORMAL_CONTAMINATED_DATA_PATH = Path(PATHS["silver_subsets_normal_contaminated_data_path"])

SILVER_ARTIFACTS_PATH = Path(PATHS["silver_artifacts_dir"])
SILVER_EDA_ARTIFACTS_PATH = Path(PATHS["silver_eda_artifacts_dir"])

GOLD_DATA_PATH = Path(PATHS["gold_preprocessed_data_path"])
GOLD_TRAIN_DATA_PATH = Path(PATHS["gold_train_data_path"])
GOLD_TEST_DATA_PATH = Path(PATHS["gold_test_data_path"])
GOLD_FIT_DATA_PATH = Path(PATHS["gold_fit_data_path"])

GOLD_PRESCALED_DATA_PATH = Path(PATHS["gold_preprocessed_prescaled_data_path"])
GOLD_SCALED_DATA_PATH = Path(PATHS["gold_preprocessed_scaled_data_path"])

GOLD_ARTIFACTS_PATH = Path(PATHS["gold_artifacts_dir"])

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

FEATURE_REGISTRY_PATH = Path(PATHS["feature_registry_path"])
IMPUTE_RECOMMENDATION_PATH = Path(PATHS["impute_recommendation_path"])

STAGE1_FEATURES_PATH = Path(PATHS["stage1_features_path"])
STAGE2_FEATURES_PATH = Path(PATHS["stage2_features_path"])
STAGE3_PRIMARY_PATH = Path(PATHS["stage3_primary_path"])
STAGE3_SECONDARY_PATH = Path(PATHS["stage3_secondary_path"])
CASCADE_DEFAULTS_RESULTS_PATH_CSV = Path(PATHS["cascade_defaults_results_path_csv"])
CASCADE_DEFAULTS_RESULTS_PATH_PICKLE = Path(PATHS["cascade_defaults_results_path_pickle"])
CASCADE_TUNED_RESULTS_PATH_CSV = Path(PATHS["cascade_tuned_results_path_csv"])
CASCADE_TUNED_RESULTS_PATH_PICKLE = Path(PATHS["cascade_tuned_results_path_pickle"])
COMPARISON_PATH = Path(PATHS["comparison_path"])

PREPROCESSING_SUMMARY_PATH = Path(PATHS["preprocessing_summary_path"])
PREPROCESSING_METADATA_PATH = Path(PATHS["preprocessing_metadata_path"])
REFERENCE_PROFILE_PATH = Path(PATHS["reference_profile_path"])

LOGS_PATH = Path(PATHS["logs_root"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# W&B
set_wandb_dir_from_config(CONFIG)


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# Path failsafes
GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  


----

In [29]:
# =========================================================
# Gold preprocessing artifact directories
# =========================================================

GOLD_PREPROCESSING_ARTIFACT_DIRS = build_artifact_dirs_from_config(
    config=CONFIG,
    stage_key="gold_preprocessing",
)

GOLD_ARTIFACTS_PATH = GOLD_PREPROCESSING_ARTIFACT_DIRS["stage_dataset_root"]
GOLD_PREPROCESSING_ROOT = GOLD_PREPROCESSING_ARTIFACT_DIRS["root"]

GOLD_FEATURE_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["features"]
GOLD_PROFILE_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["profiles"]
GOLD_MODEL_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["models"]
GOLD_SUMMARY_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["summaries"]
GOLD_METADATA_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["metadata"]
GOLD_CONFIG_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["config"]
GOLD_LINEAGE_DIR = GOLD_PREPROCESSING_ARTIFACT_DIRS["lineage"]

STAGE1_FEATURES_PATH = GOLD_FEATURE_DIR / FILENAMES["stage1_features_file_name"]
STAGE2_FEATURES_PATH = GOLD_FEATURE_DIR / FILENAMES["stage2_features_file_name"]
STAGE3_PRIMARY_PATH = GOLD_FEATURE_DIR / FILENAMES["stage3_primary_file_name"]
STAGE3_SECONDARY_PATH = GOLD_FEATURE_DIR / FILENAMES["stage3_secondary_file_name"]

REFERENCE_PROFILE_PATH = GOLD_PROFILE_DIR / FILENAMES["reference_profile_file_name"]

PREPROCESSING_SUMMARY_PATH = (
    GOLD_SUMMARY_DIR
    / FILENAMES["preprocessing_summary_file_name"]
)

PREPROCESSING_METADATA_PATH = (
    GOLD_METADATA_DIR
    / FILENAMES["preprocessing_metadata_file_name"]
)

CONFIG_SNAPSHOT_PATH = (
    GOLD_CONFIG_DIR
    / f"{DATASET_NAME}__gold_preprocessing__resolved_config.yaml"
)

gold_preprocesssing_ledger_path = (
    GOLD_LINEAGE_DIR
    / FILENAMES["gold_preprocessing_ledger_file_name"]
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

---

In [30]:
# =============================================================================
# SQL Runtime Context
# Purpose:
#   Establish the Postgres connection and resolve the dataset/run identifiers
#   used by SQL logging and medallion table writes.
# =============================================================================

engine = get_engine_from_env()

CAPSTONE_SCHEMA = os.getenv("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = os.getenv(
    "SYNTHETIC_DATASET_ID",
    globals().get("DATASET_NAME", "pump_synthetic_v1"),
)

RUN_ID = os.getenv(
    "SYNTHETIC_RUN_ID",
    globals().get("RUN_ID", "synthetic_run_001"),
)

print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")


SQL schema: capstone
Dataset ID: pump_synthetic_v1
Run ID: synthetic_run_001


In [31]:

# =============================================================================
# SQL Smoke Check
# Purpose:
#   Confirm the database connection and expected capstone schemas/tables exist.
# =============================================================================

sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


----

## Start Logging for the Gold Preprocessing Stage

Before I begin changing the data, I want logging turned on so this notebook records what happened during the run.

This gives me a cleaner record of the workflow than notebook output alone. It also helps with debugging later if I need to confirm what file paths were used, what stage ran, or where a failure happened.

In [32]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_preprocessing.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)


2026-06-02 13:49:45,872 | INFO | capstone.gold | Gold stage starting
2026-06-02 13:49:45,875 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-06-02 13:49:45,877 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-06-02 13:49:45,879 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-02 13:49:45,881 | INFO | capstone.gold | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-02 13:49:45,884 | INFO | capstone.gold | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-02 13:49:45,886 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-06-02 13:49:45,889 | INFO | capstone.gold | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-06-02 13:49:45,893 | INFO | capstone.gold | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-06-02 13:49:45,896 | INFO | capstone.gold | Previous Layer (Silver) Testing Path Loaded: /workspace/data/silver/tes

----

## Initialize the W&B Run

Here I start the Weights & Biases run for the Gold preprocessing stage.

I am using W&B here as run tracking rather than as part of the actual data transformation. It helps document the settings, artifacts, and outputs tied to this Gold run so the preprocessing stage is easier to trace and compare later.

In [33]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_preprocessing",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "silver_path": str(SILVER_TRAIN_DATA_PATH),
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "impute_recommendation_path": str(IMPUTE_RECOMMENDATION_PATH),
        "gold_output_path": str(GOLD_DATA_PATH),
        "scaler_kind": str(SCALER_KIND),
        "stage2_target_feature_count": int(STAGE2_TARGET_FEATURE_COUNT),
        "stage3_primary_count": int(STAGE3_PRIMARY_COUNT),
        "stage3_secondary_count": int(STAGE3_SECONDARY_COUNT),
    },
)
logger.info("W&B initialized: %s", wandb_run.name)


2026-06-02 13:49:49,910 | INFO | capstone.gold | W&B initialized: gold__001


----

## Initialize the Gold Ledger

This notebook also uses a ledger so I can keep a structured record of the major steps taken during Gold preprocessing.

I treat the ledger as a cleaner audit trail of the notebook. Instead of relying only on scattered notebook prints and logs, I also capture the important actions in a structured format that is easier to review later.

In [34]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-06-02 13:49:50,289 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-06-02T13:49:50.289274+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-06-02T13:49:50.289274+00:00',
 'stage': 'gold',
 'recipe': 'gold_preprocessing__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

----

## Load the Silver Input and Supporting Artifacts

This is the point where I load the finalized Silver dataset into the Gold preprocessing workflow.

I also load the supporting artifacts that tell Gold how to prepare the data, including:
- the feature registry
- the imputation recommendation
- the Silver-based feature list and feature set ID

So this step is doing more than just loading a parquet file. It is also bringing in the upstream decisions that Gold needs in order to stay aligned with the earlier stages of the pipeline.

In [35]:
# =========================================================
# Load profiled Silver subset outputs for Gold preprocessing
# =========================================================

USE_PROFILED_SILVER_SUBSETS = True

if USE_PROFILED_SILVER_SUBSETS:
    profiled_silver_path = SILVER_PROFILED_DATAFRAME_DATA_PATH
    normal_clean_path = SILVER_NORMAL_CLEAN_DATA_PATH
    normal_contaminated_path = SILVER_NORMAL_CONTAMINATED_DATA_PATH

    if not profiled_silver_path.exists():
        raise FileNotFoundError(
            "Profiled Silver subset dataframe was not found.\n"
            f"Expected: {profiled_silver_path}\n"
            "Run Silver 02a subset builder first."
        )

    if not normal_clean_path.exists():
        raise FileNotFoundError(
            "Normal-clean subset was not found.\n"
            f"Expected: {normal_clean_path}\n"
            "Run Silver 02a subset builder first."
        )

    silver_path = profiled_silver_path

    logger.info("Loading profiled Silver parquet: %s", silver_path)
    silver_dataframe = pd.read_parquet(silver_path)

    logger.info("Loading Gold normal-clean subset: %s", normal_clean_path)
    normal_clean_dataframe = pd.read_parquet(normal_clean_path)

    if normal_contaminated_path.exists():
        logger.info("Loading normal-contaminated subset: %s", normal_contaminated_path)
        normal_contaminated_dataframe = pd.read_parquet(normal_contaminated_path)
    else:
        logger.warning("Normal-contaminated subset not found: %s", normal_contaminated_path)
        normal_contaminated_dataframe = pd.DataFrame()

    if "machine_status__profiled" not in silver_dataframe.columns:
        raise KeyError(
            "Expected 'machine_status__profiled' in profiled Silver dataframe, but it was not found."
        )

    gold_train_normal_mask = silver_dataframe["machine_status__profiled"].eq("normal_clean")
    gold_train_normal_dataframe = silver_dataframe.loc[gold_train_normal_mask].copy()

    gold_contaminated_normal_mask = silver_dataframe["machine_status__profiled"].eq("normal_contaminated")
    gold_contaminated_normal_dataframe = silver_dataframe.loc[gold_contaminated_normal_mask].copy()

else:
    silver_path = SILVER_TRAIN_DATA_PATH

    logger.info("Loading original Silver parquet: %s", silver_path)
    silver_dataframe = load_data(silver_path)

    normal_clean_dataframe = pd.DataFrame()
    normal_contaminated_dataframe = pd.DataFrame()
    gold_train_normal_dataframe = pd.DataFrame()
    gold_contaminated_normal_dataframe = pd.DataFrame()
    gold_train_normal_mask = pd.Series(False, index=silver_dataframe.index)
    gold_contaminated_normal_mask = pd.Series(False, index=silver_dataframe.index)

logger.info("Silver shape=%s", silver_dataframe.shape)

ledger.add(
    kind="step",
    step="load_profiled_silver_inputs",
    message="Loaded profiled Silver parquet and subset outputs for Gold preprocessing.",
    data={
        "silver_path": str(silver_path),
        "profiled_silver_path": str(profiled_silver_path) if USE_PROFILED_SILVER_SUBSETS else None,
        "normal_clean_path": str(normal_clean_path) if USE_PROFILED_SILVER_SUBSETS else None,
        "normal_contaminated_path": str(normal_contaminated_path) if USE_PROFILED_SILVER_SUBSETS else None,
        "shape": {
            "rows": int(len(silver_dataframe)),
            "columns": int(len(silver_dataframe.columns)),
        },
        "normal_clean_rows": int(len(normal_clean_dataframe)) if USE_PROFILED_SILVER_SUBSETS else None,
        "gold_train_normal_rows": int(len(gold_train_normal_dataframe)) if USE_PROFILED_SILVER_SUBSETS else None,
        "normal_contaminated_rows": int(len(normal_contaminated_dataframe)) if USE_PROFILED_SILVER_SUBSETS else None,
    },
    logger=logger,
)

display(silver_dataframe.head(3))

2026-06-02 13:49:50,673 | INFO | capstone.gold | Loading profiled Silver parquet: /workspace/data/silver/train/pump__silver_subsets__profiled_dataframe.parquet
2026-06-02 13:49:55,074 | INFO | capstone.gold | Loading Gold normal-clean subset: /workspace/data/silver/train/pump__silver_subsets__normal_clean.parquet
2026-06-02 13:49:58,403 | INFO | capstone.gold | Loading normal-contaminated subset: /workspace/data/silver/train/pump__silver_subsets__normal_contaminated.parquet
2026-06-02 13:50:04,581 | INFO | capstone.gold | Silver shape=(220320, 339)
2026-06-02 13:50:04,586 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-06-02T13:50:04.586921+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'load_profiled_silver_inputs', 'message': 'Loaded profiled Silver parquet and subset outputs for Gold preprocessing.', 'why': None, 'consequence': None, 'data': {'silver_path': '/workspace/data/silver/train/pump__silver_subsets__profiled_dataframe.parqu

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_43__value_deviation,sensor_43__delta_deviation,sensor_43__value_abnormal_flag,sensor_43__delta_abnormal_flag,sensor_43__any_abnormal_flag,sensor_44__value_deviation,sensor_44__delta_deviation,sensor_44__value_abnormal_flag,sensor_44__delta_abnormal_flag,sensor_44__any_abnormal_flag,sensor_45__value_deviation,sensor_45__delta_deviation,sensor_45__value_abnormal_flag,sensor_45__delta_abnormal_flag,sensor_45__any_abnormal_flag,sensor_46__value_deviation,sensor_46__delta_deviation,sensor_46__value_abnormal_flag,sensor_46__delta_abnormal_flag,sensor_46__any_abnormal_flag,sensor_47__value_deviation,sensor_47__delta_deviation,sensor_47__value_abnormal_flag,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-02 04:50:00.299056+00:00,65fc2fb9e5fc2892d89db066e4321189eb738065762b29...,batch,14598431322315673869,run__001,sensor.csv,0,unsplit,0f2c8895d670e24b98d928dd04011749d253b335098ac1...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,...,0.131868,NaN,False,False,False,0.076923,NaN,False,False,False,4.932429,NaN,False,False,False,0.980392,NaN,False,False,False,0.739131,NaN,False,False,False,0.716570,NaN,False,False,False,2.780487,NaN,False,False,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-02 04:50:00.299056+00:00,65fc2fb9e5fc2892d89db066e4321189eb738065762b29...,batch,15954729095895098000,run__001,sensor.csv,1,unsplit,0f2c8895d670e24b98d928dd04011749d253b335098ac1...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,...,0.131868,0.000000,False,False,False,0.076923,0.000000,False,False,False,4.932429,0.000000,False,False,False,0.980392,0.000000,False,False,False,0.739131,0.000000,False,False,False,0.716570,0.000000,False,False,False,2.780487,0.000000,False,False,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-02 04:50:00.299056+00:00,65fc2fb9e5fc2892d89db066e4321189eb738065762b29...,batch,100417032970908

## Resolve the Parent Truth Record and Confirm the Dataset Identity

Before I start building Gold outputs, I want to confirm what Silver truth record this dataframe came from and what dataset identity should carry forward.

This matters because Gold should inherit its lineage from Silver rather than creating disconnected outputs. In this step I verify:
- the parent truth hash
- the dataset name
- the pipeline mode
- the correct artifact locations that should be used for this run

That way the Gold stage stays tied back to the upstream Silver artifact in a traceable way.

In [36]:
GOLD_PARENT_TRUTH_HASH = extract_truth_hash(silver_dataframe)

if GOLD_PARENT_TRUTH_HASH is None:
    raise ValueError("Gold preprocessing input dataframe does not contain a readable meta__truth_hash value.")

SILVER_DATASET_NAME = (
    silver_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)

SILVER_DATASET_NAME = SILVER_DATASET_NAME[SILVER_DATASET_NAME != ""]

if len(SILVER_DATASET_NAME) == 0:
    raise ValueError("Gold preprocessing input dataframe is missing usable meta__dataset values.")

SILVER_DATASET_NAME = str(SILVER_DATASET_NAME.iloc[0]).strip()

silver_truth = load_parent_truth_record_from_dataframe(
    dataframe=silver_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="silver",
    dataset_name=SILVER_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(silver_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(silver_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(silver_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

FEATURE_REGISTRY_PATH = Path(get_artifact_path_from_truth(silver_truth, "feature_registry_dir")) / "preeda/registry" / f"{DATASET_NAME}__silver__feature_registry.json"

silver_eda_artifacts_dir = Path(PATHS["artifacts_root"]) / "silver_eda" / DATASET_NAME
IMPUTE_RECOMMENDATION_PATH = silver_eda_artifacts_dir / FILENAMES["impute_recommendation_file_name"]

if "meta__pipeline_mode" not in silver_dataframe.columns:
    silver_dataframe["meta__pipeline_mode"] = PIPELINE_MODE
else:
    silver_dataframe["meta__pipeline_mode"] = silver_dataframe["meta__pipeline_mode"].fillna(PIPELINE_MODE)

gold_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
    process_run_id=GOLD_PROCESS_RUN_ID,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

gold_truth = update_truth_section(
    gold_truth,
    "config_snapshot",
    {
        "gold_version": GOLD_VERSION,
        "recipe_id": RECIPE_ID,
        "dataset_name_config": DATASET_NAME_CONFIG,
        "dataset_name_parent_truth": DATASET_NAME,
        "train_fraction": TRAIN_FRACTION,
        "random_seed": RANDOM_SEED,
        "scaler_kind": SCALER_KIND,
        "stage2_target_feature_count": STAGE2_TARGET_FEATURE_COUNT,
        "stage3_primary_count": STAGE3_PRIMARY_COUNT,
        "stage3_secondary_count": STAGE3_SECONDARY_COUNT,
        "pipeline_mode": PIPELINE_MODE,
    },
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "parent_layer_name": "silver",
        "parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "dataset_name_from_parent_truth": DATASET_NAME,
    },
)

logger.info("Resolved Silver parent truth hash: %s", GOLD_PARENT_TRUTH_HASH)
logger.info("Resolved Gold preprocessing dataset name from Silver truth: %s", DATASET_NAME)

print("Gold preprocessing parent truth hash:", GOLD_PARENT_TRUTH_HASH)
print("Gold preprocessing dataset name from parent truth:", DATASET_NAME)
print("Feature registry path from Silver truth:", FEATURE_REGISTRY_PATH)
print("Impute recommendation path from Silver EDA artifacts:", IMPUTE_RECOMMENDATION_PATH)

2026-06-02 13:50:05,868 | INFO | capstone.gold | Resolved Silver parent truth hash: 0f2c8895d670e24b98d928dd04011749d253b335098ac1e14d0dba518356af1e
2026-06-02 13:50:05,871 | INFO | capstone.gold | Resolved Gold preprocessing dataset name from Silver truth: pump


Gold preprocessing parent truth hash: 0f2c8895d670e24b98d928dd04011749d253b335098ac1e14d0dba518356af1e
Gold preprocessing dataset name from parent truth: pump
Feature registry path from Silver truth: /workspace/artifacts/silver/pump/preeda/preeda/registry/pump__silver__feature_registry.json
Impute recommendation path from Silver EDA artifacts: /workspace/artifacts/silver_eda/pump/imputation__recommendation.json


---

## Validate Required Silver Support Artifacts

### Ask

Why do I validate the feature registry before continuing?

### Answer

Gold preprocessing should not guess which columns are modeling features.

This cell confirms that the feature registry and supporting Silver artifacts exist before the notebook starts transforming data. If those artifacts are missing, the correct fix is to rerun the upstream Silver notebooks rather than silently continuing with incomplete context.

In [37]:
# =========================================================
# Load feature registry and imputation recommendation
# after resolving Silver parent truth
# =========================================================

if not FEATURE_REGISTRY_PATH.exists():
    raise FileNotFoundError(
        "Feature registry path was resolved from Silver truth, but the file does not exist.\n"
        f"Expected: {FEATURE_REGISTRY_PATH}"
    )

logger.info(
    "Loading feature registry from resolved Silver truth path: %s",
    FEATURE_REGISTRY_PATH,
)

feature_registry_raw = load_json(FEATURE_REGISTRY_PATH)

feature_registry = require_dict(
    feature_registry_raw,
    "feature_registry",
)

feature_columns_raw = require_list(
    feature_registry.get("feature_columns"),
    "feature_registry['feature_columns']",
)

feature_columns = [
    str(column_name).strip()
    for column_name in feature_columns_raw
    if str(column_name).strip() != ""
]

feature_set_id = str(
    feature_registry.get("feature_set_id", "unknown_feature_set")
)

if len(feature_columns) == 0:
    raise ValueError(
        "Feature registry loaded successfully, but feature_columns is empty.\n"
        f"Feature registry path: {FEATURE_REGISTRY_PATH}"
    )

missing_feature_columns = [
    column_name
    for column_name in feature_columns
    if column_name not in silver_dataframe.columns
]

if missing_feature_columns:
    logger.warning(
        "Some feature registry columns are missing from the Gold input dataframe and will be skipped: %s",
        missing_feature_columns[:20],
    )

feature_columns = [
    column_name
    for column_name in feature_columns
    if column_name in silver_dataframe.columns
]

if len(feature_columns) == 0:
    raise ValueError(
        "After intersecting the feature registry with the Gold input dataframe, "
        "no usable feature columns remained."
    )

STAGE2_FEATURE_COLUMNS = normalize_feature_columns(
    feature_columns,
    "STAGE2_FEATURE_COLUMNS",
)

logger.info("Loading imputation recommendation: %s", IMPUTE_RECOMMENDATION_PATH)

imputation_recommendation_raw = load_json(
    IMPUTE_RECOMMENDATION_PATH,
    raise_if_missing=False,
    default={},
)

imputation_recommendation = require_dict(
    imputation_recommendation_raw,
    "imputation_recommendation",
)

recommended_imputation = str(
    imputation_recommendation.get(
        "recommendation",
        "forward_fill_within_group_then_median",
    )
)

ledger.add(
    kind="step",
    step="load_feature_registry_and_imputation_recommendation",
    message="Loaded feature registry and imputation recommendation after resolving Silver parent truth.",
    data={
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "feature_registry_exists": bool(FEATURE_REGISTRY_PATH.exists()),
        "feature_set_id": str(feature_set_id),
        "feature_count": int(len(STAGE2_FEATURE_COLUMNS)),
        "missing_feature_count": int(len(missing_feature_columns)),
        "missing_feature_columns_sample": missing_feature_columns[:20],
        "imputation_recommendation_path": str(IMPUTE_RECOMMENDATION_PATH),
        "recommended_imputation": str(recommended_imputation),
    },
    logger=logger,
)

print("Loaded feature registry:", FEATURE_REGISTRY_PATH)
print("Feature set ID:", feature_set_id)
print("Feature column count:", len(STAGE2_FEATURE_COLUMNS))
print("First 10 feature columns:", STAGE2_FEATURE_COLUMNS[:10])
print("Recommended imputation:", recommended_imputation)

FileNotFoundError: Feature registry path was resolved from Silver truth, but the file does not exist.
Expected: /workspace/artifacts/silver/pump/preeda/preeda/registry/pump__silver__feature_registry.json

----

## Create the Working Gold Dataframe and Start Runtime Tracking

Now I create the in-memory working dataframe that Gold preprocessing will use for the rest of the notebook.

I also stamp some run-level timing and process information into the truth record so this stage has a clear record of when it was processed and which Gold recipe and version were used.

In [ ]:
dataframe = silver_dataframe.copy()

GOLD_PROCESSED_AT_UTC = pd.Timestamp.now(tz="UTC")

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "processed_at_utc": GOLD_PROCESSED_AT_UTC,
        "gold_version": GOLD_VERSION,
        "preprocessing_recipe_id": RECIPE_ID,
        "gold_input_source": "silver_subset_profiled_dataframe" if USE_PROFILED_SILVER_SUBSETS else "silver_dataframe",
        "gold_training_normal_source": "machine_status__profiled == normal_clean" if USE_PROFILED_SILVER_SUBSETS else "original_silver_normal_logic",
    },
)

gold_working_dataframe = dataframe.copy()

## Review Profiled Silver Split Inputs

### Ask

Why check the profiled Silver shapes here?

### Answer

Since this Gold run is using profiled Silver subsets, I want to confirm the full profiled dataframe and the normal-training subset are both available.

This is a quick validation that the clean-normal workflow from Silver 02a is actually feeding Gold preprocessing.

In [ ]:
if USE_PROFILED_SILVER_SUBSETS:
    print("profiled silver shape:", silver_dataframe.shape)
    print("gold train normal shape:", gold_train_normal_dataframe.shape)
    print("gold contaminated normal shape:", gold_contaminated_normal_dataframe.shape)
    display(
        silver_dataframe["machine_status__profiled"]
        .value_counts(dropna=False)
        .rename_axis("machine_status__profiled")
        .reset_index(name="row_count")
    )

----

## Stamp Stable Gold Row Identity

Before any Gold transformations can change row order or subset membership, I want to stamp a stable row identifier onto the working dataframe. This gives every downstream modeling stage a permanent row key that can be used to attach anomaly scores, flags, and evaluation outputs back to the original Gold timeline.

In [ ]:
gold_working_dataframe = ensure_stable_row_id(
    gold_working_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="ensure_stable_row_id",
    message="Stamped stable row identity onto the Gold working dataframe before downstream transformations.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_working_dataframe)),
        "row_id_unique": bool(gold_working_dataframe["meta__row_id"].is_unique),
        "row_id_null_count": int(gold_working_dataframe["meta__row_id"].isna().sum()),
    },
    logger=logger,
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "row_tracking": {
            "row_id_column": "meta__row_id",
            "row_count": int(len(gold_working_dataframe)),
            "row_id_unique": bool(gold_working_dataframe["meta__row_id"].is_unique),
        }
    },
)

----

## Reload the Parent Truth Record for Direct Reference

Here I reload the Silver truth record directly from the working dataframe.

This may look slightly repetitive, but it keeps the parent truth easy to reference during later Gold steps. That becomes useful when Gold needs to inherit things like encoding decisions, feature context, or artifact paths from Silver.

In [ ]:
silver_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_working_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="silver",
    dataset_name=DATASET_NAME,
    column_name="meta__truth_hash",
)

----

## Define the Episode-Based Train/Test Split Logic

This function creates the logic for splitting the Gold dataframe into train and test partitions using episode-aware structure rather than a random row split.

That matters for this project because I want the split to respect the sequence and grouping behavior in the dataset. A random split could leak future behavior into training or break the structure of an anomaly episode in ways that would not match a more realistic workflow.

In [ ]:
def build_episode_based_split_mask(
    dataframe: pd.DataFrame,
    *,
    train_fraction: float,
    episode_column: str = "meta__episode_id",
    order_column: str | None = None,
) -> tuple[pd.Series, dict]:
    """
    Build a train/test mask at the episode level.

    The split is chronological by episode start order, not by raw episode id sorting.

    - Earlier episodes become train.
    - Later episodes become test.
    - Every episode belongs entirely to one split.
    - At least one episode is held out for test.
    """

    if episode_column not in dataframe.columns:
        raise ValueError(f"Episode column '{episode_column}' not found in dataframe.")

    if not 0 < float(train_fraction) < 1:
        raise ValueError(
            f"train_fraction must be between 0 and 1, not inclusive. Got: {train_fraction}"
        )

    working_dataframe = dataframe.copy()

    if working_dataframe[episode_column].isna().any():
        missing_episode_count = int(working_dataframe[episode_column].isna().sum())
        raise ValueError(
            f"{episode_column} contains {missing_episode_count} missing values. "
            "Cannot safely build an episode-level split."
        )

    if order_column is None:
        for candidate_column in ["time_index", "event_step", "meta__row_id"]:
            if candidate_column in working_dataframe.columns:
                order_column = candidate_column
                break

    if order_column is None:
        order_column = "__row_order_for_episode_split"
        working_dataframe[order_column] = np.arange(len(working_dataframe))

    if order_column not in working_dataframe.columns:
        raise ValueError(f"Order column '{order_column}' not found in dataframe.")

    episode_order = (
        working_dataframe
        .groupby(episode_column, dropna=False)[order_column]
        .min()
        .sort_values(kind="mergesort")
    )

    ordered_episodes = episode_order.index.tolist()
    n_episodes = len(ordered_episodes)

    if n_episodes < 2:
        raise ValueError(
            f"Need at least 2 episodes for train/test splitting. Found: {n_episodes}"
        )

    train_episode_count = int(np.floor(n_episodes * float(train_fraction)))
    train_episode_count = max(1, train_episode_count)
    train_episode_count = min(train_episode_count, n_episodes - 1)

    train_episodes = ordered_episodes[:train_episode_count]
    test_episodes = ordered_episodes[train_episode_count:]

    train_episode_set = set(train_episodes)
    test_episode_set = set(test_episodes)

    if train_episode_set & test_episode_set:
        raise ValueError("Episode split leakage detected: train/test episode overlap exists.")

    train_mask = working_dataframe[episode_column].isin(train_episode_set)

    split_info = {
        "train_fraction": float(train_fraction),
        "episode_column": episode_column,
        "order_column": order_column,
        "total_episodes": int(n_episodes),
        "train_episode_count": int(len(train_episodes)),
        "test_episode_count": int(len(test_episodes)),
        "train_episodes": [str(value) for value in train_episodes],
        "test_episodes": [str(value) for value in test_episodes],
        "train_rows": int(train_mask.sum()),
        "test_rows": int((~train_mask).sum()),
    }

    return train_mask, split_info

## Inspect Available Gold Columns Before Splitting

### Ask

Why display the working dataframe columns here?

### Answer

Before building the train/test split, I want to confirm the fields available for ordering and grouping.

This helps verify that expected columns like `time_index`, `event_step`, `meta__row_id`, and profiled status fields are present before the split logic runs.

In [ ]:
display(gold_working_dataframe.columns)

## Build the Train/Test Split Mask

Now I apply the episode-based split logic to the working Gold dataframe.

The goal here is to create a stable split that downstream modeling notebooks can reuse. Once this mask is created, the baseline notebook, the cascade notebook, and the comparison notebook can all work from the same train/test partition instead of each generating their own.

In [ ]:
split_order_column = None

for candidate_column in ["time_index", "event_step", "meta__row_id"]:
    if candidate_column in gold_working_dataframe.columns:
        split_order_column = candidate_column
        break

train_mask, split_info = build_episode_based_split_mask(
    gold_working_dataframe,
    train_fraction=TRAIN_FRACTION,
    episode_column="meta__episode_id",
    order_column=split_order_column,
)

ledger.add(
    kind="step",
    step="build_episode_based_split_mask",
    message="Created chronological episode-level train/test split for Gold modeling.",
    data=split_info,
    logger=logger,
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "episode_split_info": split_info,
    },
)

display(pd.DataFrame([split_info]))

### Ask

Why am I creating the split here in preprocessing instead of later inside each model notebook?

### Answer

I am doing it here so the split becomes part of the shared Gold data product rather than a notebook-specific choice.

That gives me two important benefits:
- every modeling notebook uses the same partition
- the comparison between baseline and cascade stays fair because both models are evaluated against the same train and test rows

So this step is really about consistency and reproducibility, not just convenience.

## Define a Helper to Stamp Training Metadata

This helper function writes the split result back into the dataframe as row-level metadata.

I want the train/test information attached directly to the rows because that makes the split easier to preserve when the Gold artifacts are saved and reused later. Instead of tracking the split separately, I can keep it in the dataframe itself.

In [ ]:
def stamp_training_metadata(
    dataframe: pd.DataFrame,
    train_mask: pd.Series | np.ndarray | None,
) -> pd.DataFrame:
    """
    Stamp the train/test split onto the dataframe.

    Adds:
    - meta__is_train_flag
    """

    working_dataframe = dataframe.copy()

    if train_mask is None:
        return working_dataframe

    if isinstance(train_mask, pd.Series):
        train_mask_aligned = (
            train_mask
            .reindex(working_dataframe.index)
            .fillna(False)
            .astype(bool)
        )
    else:
        if len(train_mask) != len(working_dataframe):
            raise ValueError("train_mask length does not match dataframe length.")

        train_mask_aligned = pd.Series(
            train_mask,
            index=working_dataframe.index,
            dtype=bool,
        )

    working_dataframe["meta__is_train_flag"] = train_mask_aligned.values

    return working_dataframe

## Stamp Train/Test Metadata onto the Working Dataframe

Here I apply the split metadata so each row carries its train/test assignment.

This does not change the underlying feature values. It just adds the split context directly to the dataframe so the Gold artifacts can preserve how the data was partitioned for modeling.

In [ ]:
gold_working_dataframe = stamp_training_metadata(
    gold_working_dataframe,
    train_mask=train_mask,
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "split_info": split_info,
    },
)

ledger.add(
    kind="step",
    step="add_train_flag",
    message="Stamped only row-level train flag; split metadata was written to Truth Store.",
    data=split_info,
    logger=logger,
)

----

## Define the Numeric Feature Selection Logic

This helper function filters the upstream feature list down to only the columns that are actually numeric in the current dataframe.

That matters because the core Gold modeling steps in this project are based on numeric sensor-style inputs. If a feature is not numeric, it should not be sent directly into scaling, reference profiling, or the Isolation Forest pipeline without additional handling.

In [ ]:
def select_numeric_feature_columns(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> list[str]:
    numeric_feature_columns: list[str] = []

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            numeric_feature_columns.append(feature_name)

    return numeric_feature_columns


## Select the Numeric Feature Set for Gold Modeling

Now I create the actual list of numeric feature columns that Gold preprocessing will use.

This step turns the broader upstream feature registry into the specific Gold modeling input set. From here forward, these columns become the main feature space used for imputation, scaling, reference profile creation, and stage-level feature selection.

In [ ]:
numeric_feature_columns = select_numeric_feature_columns(
    silver_dataframe,
    feature_columns=feature_columns,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger

ledger.add(
    kind="step",
    step="select_numeric_feature_columns",
    message="Capture the a list of the numeric feature columns from the dataframe",
    data={
        "silver_dataframe_path": str(silver_path),
        "silver_dataframe_shape": list(silver_dataframe.shape),
        "numeric_feature_columns": numeric_feature_columns
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


----

## Define the One-Hot Encoding Logic Based on Upstream Truth

Some datasets may still contain columns that need one-hot encoding before they are fully model-ready.

This helper uses the upstream truth information to determine whether that step is needed and, if so, which columns should be encoded. I like this approach because Gold is not guessing. It is following the encoding guidance already established upstream.

In [ ]:
def apply_one_hot_encoding_from_truths(
    gold_dataframe: pd.DataFrame,
    *,
    upstream_truth_record: dict,
    drop_first: bool = False,
    dummy_na: bool = False,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Apply one-hot encoding in Gold using the encoding instructions saved in the
    upstream truth record.

    Parameters
    ----------
    gold_dataframe : pd.DataFrame
        Input dataframe for Gold processing.
    upstream_truth_record : dict
        Truth record containing:
            - needs_one_hot_encoding
            - one_hot_encoding_columns
    drop_first : bool, default=False
        Whether to drop the first category level.
    dummy_na : bool, default=False
        Whether to create a separate indicator for missing values.

    Returns
    -------
    encoded_gold_dataframe : pd.DataFrame
        Dataframe after one-hot encoding.
    encoded_column_names : list[str]
        List of original columns that were encoded.
    """
    encoded_gold_dataframe = gold_dataframe.copy()

    needs_one_hot_encoding = bool(
        upstream_truth_record.get("needs_one_hot_encoding", False)
    )

    one_hot_encoding_columns = upstream_truth_record.get(
        "one_hot_encoding_columns",
        []
    )

    if not needs_one_hot_encoding:
        return encoded_gold_dataframe, []

    available_encoding_columns = [
        column_name
        for column_name in one_hot_encoding_columns
        if column_name in encoded_gold_dataframe.columns
    ]

    if not available_encoding_columns:
        return encoded_gold_dataframe, []

    encoded_gold_dataframe = pd.get_dummies(
        encoded_gold_dataframe,
        columns=available_encoding_columns,
        drop_first=drop_first,
        dummy_na=dummy_na,
        dtype=int,
    )

    return encoded_gold_dataframe, available_encoding_columns

## Apply One-Hot Encoding When Required

Here I apply one-hot encoding only if the upstream Silver truth says it is needed.

For this pump dataset, the main feature set is mostly numeric, so this step may not change much. But it is still useful to keep this logic in place because it makes the Gold preprocessing workflow more controlled and easier to reuse.

In [ ]:
gold_working_dataframe, applied_one_hot_encoding_columns = apply_one_hot_encoding_from_truths(
    gold_dataframe=gold_working_dataframe,
    upstream_truth_record=silver_truth,
    drop_first=False,
    dummy_na=False,
)

gold_truth["needs_one_hot_encoding"] = bool(
    silver_truth.get("needs_one_hot_encoding", False)
)
gold_truth["one_hot_encoding_columns"] = silver_truth.get(
    "one_hot_encoding_columns",
    []
)
gold_truth["applied_one_hot_encoding_columns"] = applied_one_hot_encoding_columns

----

## Define the Imputation Logic

This helper function defines how missing numeric feature values should be filled during Gold preprocessing.

In this notebook, the strategy is to use a structured imputation approach rather than a one-size-fits-all global fill. The purpose is to handle missing values in a way that respects ordering and grouping before falling back to a broader numeric fill if needed.

In [ ]:
def apply_imputation(
    dataframe: pd.DataFrame,
    *,
    numeric_feature_columns: list[str],
    method: str,
    train_mask: pd.Series | None = None,
) -> tuple[pd.DataFrame, dict]:
    """
    Apply numeric feature imputation while preserving the original row order.

    Supported methods
    -----------------
    - forward_fill_within_group_then_median
    - global_mean
    - global_median fallback
    """

    working_dataframe = dataframe.copy()

    # Preserve original row order so temporary sorting does not damage timeline order.
    working_dataframe["__original_row_order_for_imputation"] = np.arange(len(working_dataframe))

    # Validate feature columns.
    missing_feature_columns = [
        feature_name
        for feature_name in numeric_feature_columns
        if feature_name not in working_dataframe.columns
    ]

    if missing_feature_columns:
        raise KeyError(
            "The following numeric feature columns are missing from the dataframe: "
            f"{missing_feature_columns[:20]}"
        )

    # Decide which rows define the imputation statistics.
    if train_mask is not None:
        aligned_train_mask = train_mask.reindex(working_dataframe.index).fillna(False).astype(bool)
        stats_dataframe = working_dataframe.loc[aligned_train_mask].copy()
    else:
        stats_dataframe = working_dataframe.copy()

    grouping_columns: list[str] = []
    ordering_column = None
    applied_method = method

    # =========================================================
    # Method: Forward-fill within asset/run group, then median
    # =========================================================
    if method == "forward_fill_within_group_then_median":
        if "meta__asset_id" in working_dataframe.columns:
            grouping_columns.append("meta__asset_id")

        if "meta__run_id" in working_dataframe.columns:
            grouping_columns.append("meta__run_id")

        if "event_step" in working_dataframe.columns:
            ordering_column = "event_step"
        elif "time_index" in working_dataframe.columns:
            ordering_column = "time_index"

        if len(grouping_columns) > 0 and ordering_column is not None:
            working_dataframe = (
                working_dataframe
                .sort_values(grouping_columns + [ordering_column])
                .reset_index(drop=True)
            )

            for feature_name in numeric_feature_columns:
                working_dataframe[feature_name] = (
                    working_dataframe
                    .groupby(grouping_columns, dropna=False)[feature_name]
                    .ffill()
                )

        for feature_name in numeric_feature_columns:
            median_value = stats_dataframe[feature_name].median(skipna=True)

            if pd.isna(median_value):
                raise ValueError(
                    f"Cannot impute '{feature_name}' because its median is NaN. "
                    "This usually means the feature is entirely missing in the statistics rows."
                )

            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(float(median_value))

    # =========================================================
    # Method: Global mean
    # =========================================================
    elif method == "global_mean":
        grouping_columns = []
        ordering_column = None

        for feature_name in numeric_feature_columns:
            mean_value = stats_dataframe[feature_name].mean(skipna=True)

            if pd.isna(mean_value):
                raise ValueError(
                    f"Cannot impute '{feature_name}' because its mean is NaN. "
                    "This usually means the feature is entirely missing in the statistics rows."
                )

            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(float(mean_value))

    # =========================================================
    # Fallback Method: Global median
    # =========================================================
    else:
        applied_method = "global_median"
        grouping_columns = []
        ordering_column = None

        for feature_name in numeric_feature_columns:
            median_value = stats_dataframe[feature_name].median(skipna=True)

            if pd.isna(median_value):
                raise ValueError(
                    f"Cannot impute '{feature_name}' because its median is NaN. "
                    "This usually means the feature is entirely missing in the statistics rows."
                )

            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(float(median_value))

    # Restore original row order for every method.
    working_dataframe = (
        working_dataframe
        .sort_values("__original_row_order_for_imputation")
        .drop(columns=["__original_row_order_for_imputation"])
        .reset_index(drop=True)
    )

    imputation_info = {
        "imputation_method": applied_method,
        "requested_method": method,
        "grouping_columns": grouping_columns,
        "ordering_column": ordering_column,
        "numeric_feature_count": int(len(numeric_feature_columns)),
        "stats_row_count": int(len(stats_dataframe)),
    }

    return working_dataframe, imputation_info

## Apply Numeric Feature Imputation

Now I run the imputation step on the numeric feature columns.

The current approach uses forward fill within the relevant grouping and ordering logic first, then falls back to median values when needed. The reason I like this strategy is that it keeps the fill process more grounded in the actual time-ordered structure of the data instead of immediately using one global value for everything.

In [ ]:
train_mask_for_stats = gold_working_dataframe["meta__is_train_flag"].astype(bool)

gold_working_dataframe, imputation_info = apply_imputation(
    gold_working_dataframe,
    numeric_feature_columns=numeric_feature_columns,
    method="forward_fill_within_group_then_median",
    train_mask=train_mask_for_stats,
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "imputation_info": imputation_info,
        "recommended_imputation": recommended_imputation,
    },
)

### Ask

Why am I using a structured fill approach here instead of just one global median from the start?

### Answer

A pure global median is simple, but it ignores the local structure of the data.

In this project, I want missing values to be handled in a way that still respects the sequence of observations when possible. So the first step is to use the nearby ordered context, and only after that do I fall back to a broader median value.

That gives me a more practical middle ground between preserving local behavior and still making sure the final dataset is complete enough for modeling.

----

## Rebuild the Training Mask After Imputation

After the imputation step, I rebuild the training mask from the stamped metadata column.

I do this because index alignment may change during transformation work. Rebuilding the mask here makes sure later steps like scaling and fit-subset creation are still using the correct train/test assignments.

In [ ]:
# Rebuild a fresh Series mask from the stamped column 
# We have to do this because the index may have changed after imputation

train_mask_flag = gold_working_dataframe["meta__is_train_flag"].astype(bool)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger


train_mask_flag_dict = train_mask_flag.to_dict()

ledger.add(
    kind="step",
    step="training_mask_flag",
    message="Creating a mask from the meta__is_train_flag boolean flag",
    data={
        "training_mask_flag_count": train_mask_flag.shape,
        "training_mask_flag_values": train_mask_flag.values,
    },
    logger=logger,
)


#### #### #### #### #### #### #### #### 



----

## Freeze a Prescaled Copy of the Gold Dataframe

At this point the dataframe has been prepared through split assignment, feature selection, and imputation, but it has not been scaled yet.

So here I keep a prescaled copy in memory. This is useful because later I want both versions:
- a cleaned **prescaled** Gold dataframe
- a cleaned **scaled** Gold dataframe

Keeping both makes the outputs more flexible for downstream use and review.

In [ ]:
gold_preprocessed_prescaled_dataframe = gold_working_dataframe.copy()

ledger.add(
    kind="step",
    step="prepare_gold_preprocessed_prescaled_dataframe",
    message="Prepared Gold prescaled dataframe in memory. Save is deferred until truth lineage is finalized.",
    data={
        "gold_preprocessed_prescaled_shape": list(gold_preprocessed_prescaled_dataframe.shape),
    },
    logger=logger,
)

display(gold_preprocessed_prescaled_dataframe.shape)

----

## Define the Scaler Factory

This helper function gives the notebook a controlled way to choose which scaler should be used.

Instead of hard-coding one scaler everywhere, I can select the scaler type through configuration and keep the scaling step more reusable. That also makes the preprocessing stage easier to compare if I ever want to test a different scaling approach later.

In [ ]:
def make_scaler(kind: str = "robust"):
    kind = kind.lower()
    if kind == "standard":
        return StandardScaler()
    elif kind == "minmax":
        return MinMaxScaler()
    elif kind == "robust":
        return RobustScaler()
    else:
        raise ValueError(f"Unknown scaler kind: {kind}. Use 'standard', 'minmax', or 'robust'.")

## Define the Scaling Workflow

This helper function handles the full scaling process:
- choose the scaler
- fit it on the selected training rows
- transform the feature columns
- save the scaler artifact
- log the scaling details

The important design choice here is that the scaler is fit on the proper training subset instead of on the full dataset. That helps avoid leakage and keeps the Gold preprocessing aligned with a more realistic modeling workflow.

In [ ]:
def fit_and_apply_scaler(
    dataframe: pd.DataFrame,
    feature_columns: Sequence[str],
    train_mask: pd.Series,
    normal_only_mask: Optional[pd.Series],
    scaler_kind: str,
    artifacts_path: Path,
    dataset_name: str,
    ledger,
) -> Tuple[pd.DataFrame, Path]:
    """
    Fit a scaler on normal-only train rows and apply it to all rows.
    """

    if normal_only_mask is not None:
        fit_mask = train_mask & normal_only_mask
        fit_source = "train ∩ normal-only"
    else:
        fit_mask = train_mask
        fit_source = "train (no explicit normal-only mask)"

    fit_rows = dataframe.loc[fit_mask, feature_columns]

    if fit_rows.empty:
        raise ValueError(
            "No rows available to fit the scaler. "
            "Check your train_mask and normal_only_mask."
        )

    scaler = make_scaler(kind=scaler_kind)
    scaler.fit(fit_rows.values)

    scaled_dataframe = dataframe.copy()
    scaled_dataframe.loc[:, feature_columns] = scaler.transform(
        scaled_dataframe[feature_columns].values
    )

    artifacts_path.mkdir(parents=True, exist_ok=True)
    
    scaler_filename = SCALER_ARTIFACT_NAME_TEMPLATE.format(
        dataset=dataset_name,
        scaler_kind=scaler_kind.lower(),
    )

    scaler_path = artifacts_path / scaler_filename
    joblib.dump(scaler, scaler_path)

    ledger.add(
        kind="transform",
        step="gold_scaling",
        message="Scaling Gold",
        data={
            "scaler_kind": scaler_kind.lower(),
            "fit_source": fit_source,
            "fit_row_count": int(fit_rows.shape[0]),
            "feature_count": int(len(feature_columns)),
            "feature_columns": list(feature_columns),
            "scaler_path": str(scaler_path),
        },
        logger=logger,
    )

    if wandb.run is not None:
        wandb.config.update(
            {
                "gold_scaler.kind": scaler_kind.lower(),
                "gold_scaler.feature_count": int(len(feature_columns)),
            },
            allow_val_change=True,
        )
        wandb.log(
            {
                "gold_scaler.fit_rows": int(fit_rows.shape[0]),
            }
        )

    return scaled_dataframe, scaler_path

## Scale the Gold Feature Set

Now I apply scaling to the Gold feature columns.

In this notebook, the scaler is fit using the training portion of the data and, when available, restricted further to the normal-only training rows. That choice is especially important for an unsupervised anomaly-detection workflow because I want the scaling reference to reflect normal operating behavior as closely as possible.

In [ ]:
normal_only_mask = (
    gold_working_dataframe["machine_status__profiled"].eq("normal_clean")
    if "machine_status__profiled" in gold_working_dataframe.columns
    else (
        (gold_working_dataframe["anomaly_flag"] == 0)
        if "anomaly_flag" in gold_working_dataframe.columns
        else None
    )
)

gold_preprocessed_scaled_dataframe, scaler_path = fit_and_apply_scaler(
    dataframe=gold_working_dataframe,
    feature_columns=numeric_feature_columns,
    train_mask=train_mask_flag,
    normal_only_mask=normal_only_mask,
    scaler_kind=SCALER_KIND,
    artifacts_path=GOLD_MODEL_DIR,
    dataset_name=DATASET_NAME,
    ledger=ledger,
)

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "feature_set_id": feature_set_id,
        "numeric_feature_count": int(len(numeric_feature_columns)),
        "scaler_path": str(scaler_path),
        "scaler_kind_runtime": SCALER_KIND,
        "recommended_imputation": recommended_imputation,
        "scaler_fit_normal_source": (
            "train ∩ machine_status__profiled == normal_clean"
            if "machine_status__profiled" in gold_working_dataframe.columns
            else "train ∩ anomaly_flag == 0"
        ),
    },
)

print(f"Scaler saved to: {scaler_path}")
print(f"Scaled dataframe shape: {gold_preprocessed_scaled_dataframe.shape}")

gold_build_info = {
    "numeric_feature_count": int(len(numeric_feature_columns)),
    "feature_set_id": str(feature_set_id),
    "recommended_imputation": str(recommended_imputation),
    "scaler": str(SCALER_KIND),
}

ledger.add(
    kind="step",
    step="build_gold_model_ready_dataframe",
    message="Built Gold model-ready dataframe. Runtime build details were written to Truth Store.",
    data=gold_build_info,
    logger=logger,
)

display(gold_build_info)

### Ask

Why fit the scaler on training data, and preferably on normal-only training data, instead of using every row?

### Answer

Because I want the Gold preprocessing to reflect the same information boundary that the model should have.

If I fit on all rows, then the scaling step would also be influenced by test data and anomalous behavior. That can blur the distinction between normal and abnormal patterns before the model even starts.

So the goal here is to make the scaled feature space anchored to the normal training behavior rather than to the full mixed dataset.

----

## Define the Normal-Only Fit Subset Logic

This helper function creates the training subset that will actually be used for unsupervised model fitting.

Since the project is centered on anomaly detection, I do not want the model fit subset to include abnormal rows when the anomaly label is available. This keeps the fit subset closer to a normal baseline.

In [ ]:
def get_training_rows_for_unsupervised_model(
    dataframe: pd.DataFrame,
    *,
    train_mask: pd.Series,
) -> pd.DataFrame:
    training_subset = dataframe.loc[train_mask].copy()

    if "machine_status__profiled" in training_subset.columns:
        training_subset = training_subset[
            training_subset["machine_status__profiled"] == "normal_clean"
        ].copy()
    elif "anomaly_flag" in training_subset.columns:
        training_subset = training_subset[training_subset["anomaly_flag"] == 0].copy()

    return training_subset

## Build the Normal-Only Fit Subset

Now I create the fit subset by filtering the training rows down to normal behavior only.

This dataset is important because it becomes the cleanest reference subset for the unsupervised modeling steps. It is also the dataframe I use for later reference-profile creation and stage-level feature selection.

In [ ]:
# Normal Only
training_rows_for_fit = get_training_rows_for_unsupervised_model(
    gold_preprocessed_scaled_dataframe,
    train_mask=train_mask_flag,
)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="get_training_rows_for_unsupervised_model",
    message="Filtered the training split to generate the normal-only Gold fit subset.",
    data={
        "gold_preprocessed_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_normal_only_split_shape": list(training_rows_for_fit.shape),
        "normal_source": (
            "machine_status__profiled == normal_clean"
            if "machine_status__profiled" in gold_preprocessed_scaled_dataframe.columns
            else "anomaly_flag == 0"
        ),
    },
    logger=logger,
)

#### #### #### #### #### #### #### ####

----

## Define the Reference Profile Logic

This helper function builds a compact reference profile from the fit subset.

For each selected feature, I summarize the center and spread of the normal training behavior using values like:
- median
- mean
- standard deviation
- lower bound
- upper bound

This profile becomes useful later because it gives the project a stable description of normal feature behavior that can support downstream rule or profile-based logic.

In [ ]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    subset_mask: Optional[pd.Series] = None,
) -> pd.DataFrame:
    """
    Build the notebook-style reference profile used by the Gold cascade notebooks.

    Output columns
    --------------
    - feature_name
    - median_value
    - mean_value
    - standard_deviation
    - lower_bound
    - upper_bound
    """
    feature_columns = [column_name for column_name in feature_columns if column_name in dataframe.columns]

    if subset_mask is None:
        profiled_dataframe = dataframe[feature_columns].copy()
    else:
        subset_mask = subset_mask.astype(bool).reindex(dataframe.index)
        profiled_dataframe = dataframe.loc[subset_mask, feature_columns].copy()

    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = pd.to_numeric(profiled_dataframe[feature_name], errors="coerce")

        reference_rows.append(
            {
                "feature_name": feature_name,
                "median_value": float(feature_series.median()) if feature_series.notna().any() else 0.0,
                "mean_value": float(feature_series.mean()) if feature_series.notna().any() else 0.0,
                "standard_deviation": float(feature_series.std()) if feature_series.notna().any() and not pd.isna(feature_series.std()) else 0.0,
                "lower_bound": float(feature_series.quantile(0.05)) if feature_series.notna().any() else 0.0,
                "upper_bound": float(feature_series.quantile(0.95)) if feature_series.notna().any() else 0.0,
            }
        )

    return pd.DataFrame(reference_rows)

## Build the Normal Reference Profile

Here I create the actual reference profile from the normal-only fit subset.

This gives me a feature-by-feature summary of normal operating behavior. I am not using it to make model predictions inside this notebook. I am building it here so later stages have a documented normal reference to work from.

In [ ]:
reference_profile = build_reference_profile(
    training_rows_for_fit,
    feature_columns=numeric_feature_columns,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger


ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built reference profile table from selected feature columns, capturing center, spread, and percentile bounds for downstream profile-based comparison.",
    data={
        "refernce_profile": reference_profile.to_dict('dict')
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


----

## Define the Stage 2 Feature Ranking Logic

This helper function ranks candidate features by training stability.

The idea is to identify which features behave more consistently in the normal training subset. In this notebook, that stability is measured using spread relative to the feature’s center, which helps me narrow the broader Stage 1 feature set down to a smaller Stage 2 set.

In [ ]:
def choose_stage2_features_from_training_stability(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    train_mask: Optional[pd.Series] = None,
    target_count: Optional[int] = None,
    min_non_null_ratio: float = 0.95,
    min_variance: float = 1e-12,
) -> Tuple[List[str], Dict[str, Any]] | List[str]:
    """
    Compatibility helper for both pipeline-style and notebook-style Stage 2 feature selection.

    Modes
    -----
    - Notebook style: provide `target_count` and optionally a pre-filtered training dataframe.
      Returns only the chosen feature list.
    - Pipeline style: provide `train_mask` without `target_count`.
      Returns (selected_columns, info).
    """
    feature_columns = [column_name for column_name in feature_columns if column_name in dataframe.columns]

    # Notebook-style ranking by coefficient of variation.
    if target_count is not None:
        ranking_rows: list[dict] = []

        for feature_name in feature_columns:
            feature_series = pd.to_numeric(dataframe[feature_name], errors="coerce")
            median_value = float(feature_series.median()) if feature_series.notna().any() else 0.0
            standard_deviation = float(feature_series.std()) if feature_series.notna().any() and not pd.isna(feature_series.std()) else 0.0
            coefficient_of_variation = standard_deviation / max(abs(median_value), 1e-6)

            ranking_rows.append(
                {
                    "feature_name": feature_name,
                    "median_value": median_value,
                    "standard_deviation": standard_deviation,
                    "coefficient_of_variation": coefficient_of_variation,
                }
            )

        ranking_dataframe = pd.DataFrame(ranking_rows).sort_values(
            by=["coefficient_of_variation", "standard_deviation", "feature_name"],
            ascending=[True, True, True],
        ).reset_index(drop=True)

        return ranking_dataframe["feature_name"].head(int(target_count)).tolist()

    if train_mask is None:
        raise ValueError("train_mask must be provided when target_count is not supplied.")

    train_mask = train_mask.astype(bool).reindex(dataframe.index)
    train_frame = dataframe.loc[train_mask, feature_columns].copy()

    selected_columns: List[str] = []
    rejected_columns: List[Dict[str, Any]] = []

    for column_name in feature_columns:
        series = pd.to_numeric(train_frame[column_name], errors="coerce")
        non_null_ratio = float(series.notna().mean()) if len(series) > 0 else 0.0
        variance = float(series.var()) if series.notna().any() else 0.0

        if non_null_ratio < float(min_non_null_ratio):
            rejected_columns.append(
                {
                    "column_name": column_name,
                    "reason": "low_non_null_ratio",
                    "non_null_ratio": non_null_ratio,
                }
            )
            continue

        if variance <= float(min_variance):
            rejected_columns.append(
                {
                    "column_name": column_name,
                    "reason": "low_variance",
                    "variance": variance,
                }
            )
            continue

        selected_columns.append(column_name)

    info = {
        "selected_feature_count": int(len(selected_columns)),
        "selected_features": list(selected_columns),
        "rejected_columns": rejected_columns,
        "min_non_null_ratio": float(min_non_null_ratio),
        "min_variance": float(min_variance),
    }

    return selected_columns, info

## Choose the Stage 2 Feature Set

Now I select the narrower Stage 2 feature set from the broader Stage 1 numeric features.

The main purpose of this step is to create a more focused subset for the narrower second stage of the cascade. Instead of using the full Stage 1 set again, I reduce it to the features that look more stable in the normal training data.

In [ ]:
stage1_feature_columns = list(numeric_feature_columns)

stage2_feature_columns = choose_stage2_features_from_training_stability(
    training_rows_for_fit,
    feature_columns=stage1_feature_columns,
    target_count=STAGE2_TARGET_FEATURE_COUNT,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger

ledger.add(
    kind="step",
    step="choose_stage2_features_from_training_stability",
    message="Ranked training features by stability and selected the top features with the lowest relative variability for Stage 2 modeling..",
    data={
        "dataframe_original_column_count": int(len(list(training_rows_for_fit.columns))),
        "dataframe_original_columns": list(training_rows_for_fit.columns),
        "stage1_feature_column_count": int(len(stage1_feature_columns)),
        "stage1_feature_columns": list(stage1_feature_columns),
        "stage2_feature_column_count": int(len(stage2_feature_columns)),
        "stage2_feature_columns": list(stage2_feature_columns),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


----

## Define the Stage 3 Sensor Grouping Logic

This helper function builds the Stage 3 primary and secondary sensor groups from the reference profile and the Stage 2 features.

The idea here is to move from a broader modeling feature space into a smaller rule/profile-oriented sensor set. That gives the later cascade stage a more targeted group of features for confirmation logic.

In [ ]:
def build_stage3_sensor_groups(
    reference_profile: pd.DataFrame,
    *,
    stage2_feature_columns: Sequence[str],
    primary_count: int,
    secondary_count: int,
) -> Tuple[List[str], List[str]]:
    """
    Notebook-style Stage 3 sensor selection.

    Rank Stage 2 features by lowest standard deviation, then assign the most
    stable features to the primary and secondary rule sets.
    """
    if reference_profile.empty:
        return [], []

    ranked_reference = reference_profile[
        reference_profile["feature_name"].isin(list(stage2_feature_columns))
    ].copy()

    ranked_reference = ranked_reference.sort_values(
        by=["standard_deviation", "feature_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

    primary_rule_sensors = ranked_reference["feature_name"].head(int(primary_count)).tolist()

    remaining_features = [
        feature_name
        for feature_name in ranked_reference["feature_name"].tolist()
        if feature_name not in primary_rule_sensors
    ]

    secondary_rule_sensors = remaining_features[: int(secondary_count)]

    return primary_rule_sensors, secondary_rule_sensors

## Build the Stage 3 Sensor Groups

Here I create the Stage 3 primary and secondary rule sensor lists.

These groups are based on the ranked stability information from the reference profile. In other words, the Stage 3 confirmation step is not picking sensors arbitrarily. It is selecting them from the more stable part of the already narrowed feature space.

In [ ]:
def build_gold_support_artifacts(
    scaled_dataframe: pd.DataFrame,
    *,
    selected_feature_columns: Sequence[str],
    train_mask: pd.Series,
    baseline_feature_columns: Optional[Sequence[str]] = None,
    stage2_target_feature_count: Optional[int] = None,
    stage3_primary_count: int = 5,
    stage3_secondary_count: int = 5,
) -> Dict[str, Any]:
    """
    Build downstream support artifacts for baseline and cascade using the newer
    Gold notebook structure.
    """
    train_mask = train_mask.astype(bool).reindex(scaled_dataframe.index)

    reference_profile = build_reference_profile(
        scaled_dataframe,
        feature_columns=selected_feature_columns,
        subset_mask=train_mask,
    )

    training_rows_for_fit = scaled_dataframe.loc[train_mask].copy()
    stage1_feature_columns = list(selected_feature_columns)

    if stage2_target_feature_count is not None:
        stage2_feature_columns = choose_stage2_features_from_training_stability(
            training_rows_for_fit,
            feature_columns=stage1_feature_columns,
            target_count=int(stage2_target_feature_count),
        )
        stage2_info = {
            "selection_mode": "target_count",
            "target_count": int(stage2_target_feature_count),
            "selected_features": list(stage2_feature_columns),
            "selected_feature_count": int(len(stage2_feature_columns)),
        }
    else:
        stage2_feature_columns, stage2_info = choose_stage2_features_from_training_stability(
            scaled_dataframe,
            feature_columns=selected_feature_columns,
            train_mask=train_mask,
            min_non_null_ratio=0.95,
            min_variance=1e-12,
        )

    stage3_primary_rule_sensors, stage3_secondary_rule_sensors = build_stage3_sensor_groups(
        reference_profile,
        stage2_feature_columns=STAGE2_FEATURE_COLUMNS,
        primary_count=int(stage3_primary_count),
        secondary_count=int(stage3_secondary_count),
    )

    return {
        "baseline_feature_columns": list(baseline_feature_columns or selected_feature_columns),
        "stage1_feature_columns": list(stage1_feature_columns),
        "reference_profile": reference_profile,
        "stage2_feature_columns": list(stage2_feature_columns),
        "stage2_info": stage2_info,
        "stage3_primary_rule_sensors": list(stage3_primary_rule_sensors),
        "stage3_secondary_rule_sensors": list(stage3_secondary_rule_sensors),
        "stage3_sensor_groups": {
            "primary": list(stage3_primary_rule_sensors),
            "secondary": list(stage3_secondary_rule_sensors),
        },
    }

## Define Gold Support Artifact Builder

### Ask

Why define a helper for support artifacts?

### Answer

Gold preprocessing creates several artifacts that later notebooks need, such as feature lists, reference profiles, and Stage 3 sensor groups.

This helper keeps those support artifacts organized so the modeling notebooks can load prepared inputs instead of recalculating preprocessing decisions.

In [ ]:
stage3_primary_rule_sensors, stage3_secondary_rule_sensors = build_stage3_sensor_groups(
    reference_profile,
    stage2_feature_columns=STAGE2_FEATURE_COLUMNS,
    primary_count=STAGE3_PRIMARY_COUNT,
    secondary_count=STAGE3_SECONDARY_COUNT,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger

ledger.add(
    kind="step",
    step="build_stage3_sensor_groups",
    message="Built Stage 3 sensor groups by ranking Stage 2 features on reference-profile stability and assigning the most stable features to primary and secondary rule sets.",
    data={
        "refernce_profile":  reference_profile.to_dict('dict'),
        "stage2_feature_column_count": int(len(list(stage2_feature_columns))),
        "stage2_feature_columns": list(stage2_feature_columns),
        "stage3_primary_rul_sensors": list(stage3_primary_rule_sensors),
        "stage3_primary_rul_sensors_count": int(len(list(stage3_primary_rule_sensors))),
        "stage3_secondary_rule_sensors": list(stage3_secondary_rule_sensors),
        "stage3_secondary_rule_sensors_count": int(len(list(stage3_secondary_rule_sensors))),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


----

## Save the Stage-Level Gold Artifacts

At this point I have created the main feature lists and reference artifacts needed for downstream modeling.

So this step saves those intermediate Gold preprocessing artifacts, such as:
- Stage 1 feature list
- Stage 2 feature list
- Stage 3 primary sensor list
- Stage 3 secondary sensor list
- reference profile

Saving these separately makes the later modeling notebooks simpler because they can load the prepared artifacts instead of recalculating them.

In [ ]:
save_json(stage1_feature_columns, STAGE1_FEATURES_PATH)
save_json(stage2_feature_columns, STAGE2_FEATURES_PATH)
save_json(stage3_primary_rule_sensors, STAGE3_PRIMARY_PATH)
save_json(stage3_secondary_rule_sensors, STAGE3_SECONDARY_PATH)

reference_profile.to_csv(REFERENCE_PROFILE_PATH, index=False)

wandb.save(str(STAGE1_FEATURES_PATH))
wandb.save(str(STAGE2_FEATURES_PATH))
wandb.save(str(STAGE3_PRIMARY_PATH))
wandb.save(str(STAGE3_SECONDARY_PATH))
wandb.save(str(REFERENCE_PROFILE_PATH))

feature_set_summary = {
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
}

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "stage_feature_summary": feature_set_summary,
    },
)

gold_truth = update_truth_section(
    gold_truth,
    "artifact_paths",
    {
        "reference_profile_path": str(REFERENCE_PROFILE_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)

ledger.add(
    kind="step",
    step="build_stage_feature_sets",
    message="Built feature sets for Stage 1, Stage 2, and Stage 3 and wrote the results to Truth Store.",
    data=feature_set_summary,
    logger=logger,
)

feature_set_summary

----

## Create the Final Gold Split Dataframes

Here I build the final split-specific Gold dataframes:
- full scaled Gold dataframe with split labels
- train dataframe
- test dataframe
- fit-normal-only dataframe

This step turns the earlier split logic into actual saved modeling inputs. It also makes the downstream notebooks cleaner because each notebook can read the exact partition it needs.

In [ ]:
if "meta__is_train_flag" in gold_preprocessed_scaled_dataframe.columns:
    gold_preprocessed_scaled_dataframe["meta__split"] = np.where(
        gold_preprocessed_scaled_dataframe["meta__is_train_flag"].astype(bool),
        "train",
        "test",
    )
else:
    raise ValueError(
        "meta__is_train_flag is missing from gold_preprocessed_scaled_dataframe. "
        "The train/test split was not stamped correctly."
    )

train_mask_flag = gold_preprocessed_scaled_dataframe["meta__is_train_flag"].astype(bool)

gold_train_dataframe = gold_preprocessed_scaled_dataframe.loc[train_mask_flag].copy()
gold_test_dataframe = gold_preprocessed_scaled_dataframe.loc[~train_mask_flag].copy()
gold_fit_dataframe = training_rows_for_fit.copy()

gold_train_dataframe["meta__split"] = "train"
gold_test_dataframe["meta__split"] = "test"
gold_fit_dataframe["meta__split"] = "fit_normal_only"

gold_split_summary = {
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "train_episode_count": int(gold_train_dataframe["meta__episode_id"].nunique())
    if "meta__episode_id" in gold_train_dataframe.columns
    else None,
    "test_episode_count": int(gold_test_dataframe["meta__episode_id"].nunique())
    if "meta__episode_id" in gold_test_dataframe.columns
    else None,
}

ledger.add(
    kind="step",
    step="build_final_gold_split_dataframes",
    message="Created final Gold train, test, and fit-normal-only dataframes.",
    data=gold_split_summary,
    logger=logger,
)

display(pd.DataFrame([gold_split_summary]))

### Ask

What is the difference between the train, test, and fit-normal-only Gold outputs?

### Answer

Each one serves a different purpose.

- The **train** dataframe contains the earlier portion of the Gold dataset and is the main training partition.
- The **test** dataframe contains the later holdout portion used for evaluation.
- The **fit-normal-only** dataframe is a filtered subset of the training rows where only normal behavior is kept.

That last one matters most for the unsupervised anomaly workflow because it gives the model and the reference profile a cleaner view of normal operating behavior.

## Finalize the Gold Truth Record

Before saving the final artifacts, I update the Gold truth record with the runtime details, split summary, source fingerprint, and artifact paths produced by this notebook.

This step is important because I do not want the Gold outputs to exist without context. The truth record ties the final datasets back to:
- the upstream Silver source
- the preprocessing settings
- the created artifacts
- the lineage information for this Gold run

In [ ]:
# =========================================================
# Verify episode split integrity
# =========================================================

def verify_gold_episode_split(
    full_dataframe: pd.DataFrame,
    train_dataframe: pd.DataFrame,
    test_dataframe: pd.DataFrame,
    fit_dataframe: pd.DataFrame,
    *,
    episode_column: str = "meta__episode_id",
    row_id_column: str = "meta__row_id",
) -> pd.DataFrame:
    required_columns = [episode_column, row_id_column, "meta__is_train_flag"]

    for column_name in required_columns:
        if column_name not in full_dataframe.columns:
            raise ValueError(f"Missing required split verification column: {column_name}")

    train_episode_set = set(train_dataframe[episode_column].dropna().astype(str))
    test_episode_set = set(test_dataframe[episode_column].dropna().astype(str))

    overlapping_episodes = sorted(train_episode_set & test_episode_set)

    if overlapping_episodes:
        raise ValueError(
            "Episode leakage detected. These episodes appear in both train and test: "
            f"{overlapping_episodes[:20]}"
        )

    train_row_ids = set(train_dataframe[row_id_column].dropna().astype(str))
    test_row_ids = set(test_dataframe[row_id_column].dropna().astype(str))
    fit_row_ids = set(fit_dataframe[row_id_column].dropna().astype(str))

    overlapping_train_test_rows = train_row_ids & test_row_ids
    fit_rows_outside_train = fit_row_ids - train_row_ids

    if overlapping_train_test_rows:
        raise ValueError(
            f"Row leakage detected between train and test. "
            f"Overlap count: {len(overlapping_train_test_rows)}"
        )

    if fit_rows_outside_train:
        raise ValueError(
            f"Fit dataframe contains rows outside the training split. "
            f"Count: {len(fit_rows_outside_train)}"
        )

    if "machine_status__profiled" in fit_dataframe.columns:
        bad_fit_rows = fit_dataframe.loc[
            fit_dataframe["machine_status__profiled"] != "normal_clean"
        ]

        if len(bad_fit_rows) > 0:
            raise ValueError(
                "Fit dataframe contains rows that are not machine_status__profiled == normal_clean. "
                f"Bad row count: {len(bad_fit_rows)}"
            )

    split_episode_summary = (
        full_dataframe
        .assign(
            split_label=np.where(
                full_dataframe["meta__is_train_flag"].astype(bool),
                "train",
                "test",
            )
        )
        .groupby([episode_column, "split_label"], dropna=False)
        .agg(
            row_count=(row_id_column, "count"),
            anomaly_rows=("anomaly_flag", "sum") if "anomaly_flag" in full_dataframe.columns else (row_id_column, "count"),
        )
        .reset_index()
    )

    test_anomaly_rows = (
        int(test_dataframe["anomaly_flag"].fillna(0).astype(int).sum())
        if "anomaly_flag" in test_dataframe.columns
        else None
    )

    if test_anomaly_rows == 0:
        logger.warning(
            "The test split contains zero anomaly_flag rows. "
            "Row-level precision/recall will not be meaningful."
        )

    print("Episode split verification passed.")
    print(f"Train episodes: {len(train_episode_set)}")
    print(f"Test episodes: {len(test_episode_set)}")
    print(f"Train rows: {len(train_dataframe):,}")
    print(f"Test rows: {len(test_dataframe):,}")
    print(f"Fit rows: {len(fit_dataframe):,}")
    print(f"Test anomaly rows: {test_anomaly_rows}")

    return split_episode_summary


gold_episode_split_summary = verify_gold_episode_split(
    full_dataframe=gold_preprocessed_scaled_dataframe,
    train_dataframe=gold_train_dataframe,
    test_dataframe=gold_test_dataframe,
    fit_dataframe=gold_fit_dataframe,
    episode_column="meta__episode_id",
    row_id_column="meta__row_id",
)

display(gold_episode_split_summary)

In [ ]:

gold_truth = update_truth_section(
    gold_truth,
    "runtime_facts",
    {
        "source_run_ids": (
            gold_preprocessed_scaled_dataframe["meta__run_id"].dropna().astype(str).unique().tolist()
            if "meta__run_id" in gold_preprocessed_scaled_dataframe.columns
            else []
        ),
        "split_summary": gold_split_summary,
    },
)

gold_truth = update_truth_section(
    gold_truth,
    "source_fingerprint",
    build_file_fingerprint(silver_path),
)

gold_truth = update_truth_section(
    gold_truth,
    "artifact_paths",
    {
        "silver_source_path": str(silver_path),
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "imputation_recommendation_path": str(IMPUTE_RECOMMENDATION_PATH),
        "gold_preprocessed_path": str(GOLD_PRESCALED_DATA_PATH),
        "gold_prescaled_path": str(GOLD_PRESCALED_DATA_PATH),
        "gold_scaled_path": str(GOLD_SCALED_DATA_PATH),
        "gold_train_path": str(GOLD_TRAIN_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "preprocessing_summary_path": str(PREPROCESSING_SUMMARY_PATH),
        "preprocessing_metadata_path": str(PREPROCESSING_METADATA_PATH),
        "reference_profile_path": str(REFERENCE_PROFILE_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "scaler_path": str(scaler_path),
    },
)

gold_truth = update_truth_section(
    gold_truth,
    "notes",
    {
        "purpose": "Gold preprocessing truth record",
    },
)

display(gold_truth["runtime_facts"])

## Save the Final Gold Preprocessing Outputs

Now I save the final Gold preprocessing artifacts to disk.

This includes the main model-ready outputs for downstream use:
- prescaled Gold dataframe
- scaled Gold dataframe
- train dataframe
- test dataframe
- fit-normal-only dataframe

At this point the notebook output is no longer just an in-memory working dataframe. It becomes a formal saved Gold artifact set that the modeling notebooks can load directly.

----

In [ ]:
required_row_tracking_columns = ["meta__row_id"]

for required_col in required_row_tracking_columns:
    if required_col not in gold_preprocessed_scaled_dataframe.columns:
        raise ValueError(f"{required_col} missing from gold_preprocessed_scaled_dataframe.")
    if required_col not in gold_train_dataframe.columns:
        raise ValueError(f"{required_col} missing from gold_train_dataframe.")
    if required_col not in gold_test_dataframe.columns:
        raise ValueError(f"{required_col} missing from gold_test_dataframe.")
    if required_col not in gold_fit_dataframe.columns:
        raise ValueError(f"{required_col} missing from gold_fit_dataframe.")

ledger.add(
    kind="step",
    step="validate_row_tracking_columns",
    message="Validated stable row identity across saved Gold artifacts.",
    data={
        "required_columns": required_row_tracking_columns,
        "gold_scaled_has_row_id": True,
        "gold_train_has_row_id": True,
        "gold_test_has_row_id": True,
        "gold_fit_has_row_id": True,
    },
    logger=logger,
)

----

In [ ]:

gold_meta_columns = identify_meta_columns(gold_preprocessed_scaled_dataframe)
gold_meta_columns = sorted(set(gold_meta_columns + [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]))

gold_feature_columns = identify_feature_columns(gold_preprocessed_scaled_dataframe)

gold_truth_record = build_truth_record(
    truth_base=gold_truth,
    row_count=len(gold_preprocessed_scaled_dataframe),
    column_count=gold_preprocessed_scaled_dataframe.shape[1] + 3,
    meta_columns=gold_meta_columns,
    feature_columns=gold_feature_columns,
)

GOLD_TRUTH_HASH = gold_truth_record["truth_hash"]

gold_preprocessed_prescaled_dataframe = stamp_truth_columns(
    gold_preprocessed_prescaled_dataframe,
    truth_hash=GOLD_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

gold_preprocessed_scaled_dataframe = stamp_truth_columns(
    gold_preprocessed_scaled_dataframe,
    truth_hash=GOLD_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

gold_train_dataframe = stamp_truth_columns(
    gold_train_dataframe,
    truth_hash=GOLD_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

gold_test_dataframe = stamp_truth_columns(
    gold_test_dataframe,
    truth_hash=GOLD_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

gold_fit_dataframe = stamp_truth_columns(
    gold_fit_dataframe,
    truth_hash=GOLD_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

gold_truth_path = save_truth_record(
    gold_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
)

append_truth_index(
    gold_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

print("Gold truth hash:", GOLD_TRUTH_HASH)
print("Gold truth path:", gold_truth_path)

In [ ]:

gold_preprocessed_prescaled_path = save_data(
    gold_preprocessed_prescaled_dataframe,
    GOLD_PRESCALED_DATA_PATH.parent,
    GOLD_PRESCALED_DATA_PATH.name,
)

gold_preprocessed_scaled_path = save_data(
    gold_preprocessed_scaled_dataframe,
    GOLD_SCALED_DATA_PATH.parent,
    GOLD_SCALED_DATA_PATH.name,
)

gold_train_path = save_data(
    gold_train_dataframe,
    GOLD_TRAIN_DATA_PATH.parent,
    GOLD_TRAIN_DATA_PATH.name,
)

gold_test_path = save_data(
    gold_test_dataframe,
    GOLD_TEST_DATA_PATH.parent,
    GOLD_TEST_DATA_PATH.name,
)

gold_fit_path = save_data(
    gold_fit_dataframe,
    GOLD_FIT_DATA_PATH.parent,
    GOLD_FIT_DATA_PATH.name,
)

wandb.save(str(gold_preprocessed_prescaled_path))
wandb.save(str(gold_preprocessed_scaled_path))
wandb.save(str(gold_train_path))
wandb.save(str(gold_test_path))
wandb.save(str(gold_fit_path))

ledger.add(
    kind="step",
    step="finalize_gold_truth_and_save_artifacts",
    message="Finalized Gold truth, stamped lineage columns, and saved prescaled/scaled/train/test/fit outputs.",
    data={
        "gold_truth_hash": GOLD_TRUTH_HASH,
        "gold_parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(gold_truth_path),
        "gold_prescaled_path": str(gold_preprocessed_prescaled_path),
        "gold_scaled_path": str(gold_preprocessed_scaled_path),
        "gold_train_path": str(gold_train_path),
        "gold_test_path": str(gold_test_path),
        "gold_fit_path": str(gold_fit_path),
        "pipeline_mode": PIPELINE_MODE,
        "process_run_id": GOLD_PROCESS_RUN_ID,
        **gold_split_summary,
    },
    logger=logger,
)
display(
    pd.DataFrame(
        [
            {
                "split": "train",
                "rows": int(len(gold_train_dataframe)),
                "abnormal_rows": int(gold_train_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_train_dataframe.columns else None,
                "path": str(gold_train_path),
            },
            {
                "split": "test",
                "rows": int(len(gold_test_dataframe)),
                "abnormal_rows": int(gold_test_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_test_dataframe.columns else None,
                "path": str(gold_test_path),
            },
            {
                "split": "fit_normal_only",
                "rows": int(len(gold_fit_dataframe)),
                "abnormal_rows": int(gold_fit_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_fit_dataframe.columns else None,
                "path": str(gold_fit_path),
            },
        ]
    )
)

## Save the Preprocessing Summary and Metadata Records

In addition to saving the dataframe outputs, I also save compact JSON records that summarize what this notebook produced.

These summary and metadata files are useful because they let me inspect the Gold preprocessing results without reopening the full notebook. They also make the stage easier to document and trace later.

In [ ]:
preprocessing_summary = {
    "gold_prescaled_path": str(GOLD_PRESCALED_DATA_PATH),
    "gold_scaled_path": str(GOLD_SCALED_DATA_PATH),
    "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
    "feature_count": int(len(numeric_feature_columns)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "imputation_method": str(imputation_info.get("imputation_method")),
    "grouping_columns": list(imputation_info.get("grouping_columns", [])),
    "ordering_column": imputation_info.get("ordering_column"),
    "scaler_kind": str(SCALER_KIND),
    "train_fraction": float(TRAIN_FRACTION),
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "gold_train_path": str(gold_train_path),
    "gold_test_path": str(gold_test_path),
    "gold_fit_path": str(gold_fit_path),
    "gold_truth_hash": GOLD_TRUTH_HASH,
    "gold_parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(gold_truth_path),
    "process_run_id": GOLD_PROCESS_RUN_ID,
    "gold_input_source": "silver_subset_profiled_dataframe" if USE_PROFILED_SILVER_SUBSETS else "silver_dataframe",
    "gold_training_normal_source": "machine_status__profiled == normal_clean" if USE_PROFILED_SILVER_SUBSETS else "anomaly_flag == 0",
}

save_json(preprocessing_summary, PREPROCESSING_SUMMARY_PATH)

preprocessing_metadata = {
    "recipe_id": RECIPE_ID,
    "gold_version": GOLD_VERSION,
    "dataset_name": DATASET_NAME,
    "feature_set_source": str(FEATURE_REGISTRY_PATH),
    "imputation_recommendation_source": str(IMPUTE_RECOMMENDATION_PATH),
    "gold_prescaled_output_path": str(GOLD_PRESCALED_DATA_PATH),
    "gold_scaled_output_path": str(GOLD_SCALED_DATA_PATH),
    "reference_profile_path": str(REFERENCE_PROFILE_PATH),
    "stage1_features_path": str(STAGE1_FEATURES_PATH),
    "stage2_features_path": str(STAGE2_FEATURES_PATH),
    "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
    "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    "gold_input_source": "silver_subset_profiled_dataframe" if USE_PROFILED_SILVER_SUBSETS else "silver_dataframe",
    "gold_training_normal_source": "machine_status__profiled == normal_clean" if USE_PROFILED_SILVER_SUBSETS else "anomaly_flag == 0",
}

save_json(preprocessing_metadata, PREPROCESSING_METADATA_PATH)

wandb.save(str(PREPROCESSING_SUMMARY_PATH))
wandb.save(str(PREPROCESSING_METADATA_PATH))

ledger.add(
    kind="step",
    step="save_preprocessing_outputs",
    message="Saved Gold preprocessing summary and metadata artifacts.",
    data={
        "preprocessing_summary_path": str(PREPROCESSING_SUMMARY_PATH),
        "preprocessing_metadata_path": str(PREPROCESSING_METADATA_PATH),
        "gold_truth_hash": GOLD_TRUTH_HASH,
        "gold_truth_path": str(gold_truth_path),
    },
    logger=logger,
)

{
    "preprocessing_summary_path": str(PREPROCESSING_SUMMARY_PATH),
    "preprocessing_metadata_path": str(PREPROCESSING_METADATA_PATH),
    "gold_truth_hash": GOLD_TRUTH_HASH,
    "gold_truth_path": str(gold_truth_path),
}

----

## Save the Ledger and Close the Tracking Run

This step writes the Gold preprocessing ledger to disk and then cleanly closes the experiment tracking run.

I like having an explicit wrap-up step here because it makes the stage feel complete. By this point, the main data outputs and supporting artifacts have already been created, so this section is focused on closing the run cleanly.

In [ ]:
#gold_preprocesssing_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_PREPROCESSING_LEDGER_FILE_NAME
ledger.write_json(gold_preprocesssing_ledger_path)

wandb.save(str(gold_preprocesssing_ledger_path))
wandb_run.finish()

----

## Run Final Gold Preprocessing Sanity Checks

Before I treat this notebook as finished, I run a final set of sanity checks across the saved Gold objects.

These checks help confirm that:
- the main Gold dataframes exist
- the train/test split sizes match the stored split mask
- the fit-normal-only dataframe really contains only normal rows
- the expected feature columns are present
- the truth record and truth hash values exist as expected

This is basically the last trust check before I move into the downstream modeling notebooks.

In [ ]:
# Gold Preprocessing Sanity Checks

print("Running Gold preprocessing sanity checks...\n")

assert "gold_preprocessed_scaled_dataframe" in locals(), "gold_preprocessed_scaled_dataframe is missing."
assert "gold_preprocessed_prescaled_dataframe" in locals(), "gold_preprocessed_prescaled_dataframe is missing."
assert "train_mask_flag" in locals(), "train_mask_flag is missing."
assert "numeric_feature_columns" in locals(), "numeric_feature_columns is missing."
assert "gold_train_dataframe" in locals(), "gold_train_dataframe is missing."
assert "gold_test_dataframe" in locals(), "gold_test_dataframe is missing."
assert "gold_fit_dataframe" in locals(), "gold_fit_dataframe is missing."
assert "gold_truth_record" in locals(), "gold_truth_record is missing."
assert "GOLD_TRUTH_HASH" in locals(), "GOLD_TRUTH_HASH is missing."

dataframe_scaled = gold_preprocessed_scaled_dataframe
dataframe_prescaled = gold_preprocessed_prescaled_dataframe
n_rows, n_cols = dataframe_scaled.shape

print(f"- Scaled Gold shape: {n_rows} rows x {n_cols} columns")

train_mask_flag = train_mask_flag.astype(bool)

assert len(train_mask_flag) == n_rows, "train_mask_flag length does not match Gold dataframe."
assert len(gold_train_dataframe) == train_mask_flag.sum(), "Train split row count mismatch."
assert len(gold_test_dataframe) == (~train_mask_flag).sum(), "Test split row count mismatch."

print(f"- Train rows: {len(gold_train_dataframe)}, Test rows: {len(gold_test_dataframe)}")

if "machine_status__profiled" in dataframe_scaled.columns:
    fit_mask_expected = train_mask_flag & (dataframe_scaled["machine_status__profiled"] == "normal_clean")
    assert len(gold_fit_dataframe) == fit_mask_expected.sum(), "Fit (normal-clean) row count mismatch."
    assert (gold_fit_dataframe["machine_status__profiled"] == "normal_clean").all(), (
        "Fit dataframe contains rows outside machine_status__profiled == normal_clean."
    )
    print(f"- Fit (normal-clean) rows: {len(gold_fit_dataframe)} (machine_status__profiled == normal_clean confirmed)")
elif "anomaly_flag" in dataframe_scaled.columns:
    fit_mask_expected = train_mask_flag & (dataframe_scaled["anomaly_flag"] == 0)
    assert len(gold_fit_dataframe) == fit_mask_expected.sum(), "Fit (normal-only) row count mismatch."
    assert gold_fit_dataframe["anomaly_flag"].sum() == 0, "Fit dataframe contains anomalous rows."
    print(f"- Fit (normal-only) rows: {len(gold_fit_dataframe)} (anomaly_flag == 0 confirmed)")
else:
    print("- Neither machine_status__profiled nor anomaly_flag available; skipping fit-subset state check.")

for required_column in numeric_feature_columns:
    assert required_column in dataframe_scaled.columns, (
        f"Missing numeric feature column in scaled dataframe: {required_column}"
    )

assert dataframe_scaled.index.is_unique, "Scaled dataframe index is not unique."
assert dataframe_prescaled.index.is_unique, "Prescaled dataframe index is not unique."

print("\nGold preprocessing sanity checks passed.")

### Ask

What am I really trying to confirm with these final sanity checks?

### Answer

I am checking that the Gold outputs are not only saved, but also internally consistent.

By the time I get here, the important question is no longer "did the notebook run?" The better question is: **did it produce a Gold artifact set that is safe to hand to the modeling notebooks without hidden split, scaling, or lineage problems?**

If these checks pass, then I can trust the Gold preprocessing outputs as the shared starting point for the baseline and cascade modeling stages.

----

In [ ]:
gold_preprocessed_prescaled_dataframe.shape

----

In [ ]:
gold_preprocessed_scaled_dataframe.shape

----

In [ ]:
print("anomaly_flag in full scaled df:", "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns)
print("anomaly_flag in train df:", "anomaly_flag" in gold_train_dataframe.columns)
print("anomaly_flag in test df:", "anomaly_flag" in gold_test_dataframe.columns)
print("anomaly_flag in fit df:", "anomaly_flag" in gold_fit_dataframe.columns)

print("\nfit dataframe profiled states:")
print(gold_fit_dataframe["machine_status__profiled"].value_counts(dropna=False))

print("\nfit dataframe anomaly_flag counts:")
print(gold_fit_dataframe["anomaly_flag"].value_counts(dropna=False))

print("\ntest dataframe anomaly_flag counts:")
print(gold_test_dataframe["anomaly_flag"].value_counts(dropna=False))

---

## Compare the Prescaled and Scaled Column Structures

This check compares the column layout of the prescaled and scaled Gold dataframes.

The main thing I want to confirm here is that scaling changed the feature values, not the overall schema. In other words, the scaled and prescaled outputs should stay aligned on structure even though the numeric feature values have been transformed.

In [ ]:

# Get column lists
prescaled_columns = gold_preprocessed_prescaled_dataframe.columns.tolist()
scaled_columns = gold_preprocessed_scaled_dataframe.columns.tolist()

print(f"Prescaled Columns columns: {prescaled_columns}")
print(f"Scaled columns: {scaled_columns}\n")

# Find columns unique to each
unique_to_prescaled = set(prescaled_columns) - set(scaled_columns)
unique_to_scaled = set(scaled_columns) - set(prescaled_columns)

print(f"Columns in dataframe1 but not in dataframe2: {unique_to_prescaled}")
print(f"Columns in dataframe2 but not in dataframe1: {unique_to_scaled}\n")

# Find all columns that are not in both (symmetric difference)
columns_not_in_both = set(prescaled_columns).symmetric_difference(set(scaled_columns))
print(f"All columns not present in both dataframes: {columns_not_in_both}")

----

## Verify Final Lineage Columns Across All Gold Outputs

This is the final lineage validation step for the main Gold artifacts.

Here I confirm that each saved Gold dataframe contains the required lineage columns and that those values match the saved Gold truth record. This matters because I want every major Gold artifact to remain traceable back to the exact preprocessing run that created it.

In [ ]:
required_gold_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

gold_frames_to_check = {
    "gold_preprocessed_prescaled": gold_preprocessed_prescaled_dataframe,
    "gold_preprocessed_scaled": gold_preprocessed_scaled_dataframe,
    "gold_fit": gold_fit_dataframe,
    "gold_train": gold_train_dataframe,
    "gold_test": gold_test_dataframe,
}

for frame_name, frame_value in gold_frames_to_check.items():
    missing_gold_meta_columns = [
        column_name
        for column_name in required_gold_meta_columns
        if column_name not in frame_value.columns
    ]
    if missing_gold_meta_columns:
        raise ValueError(
            f"{frame_name} is missing required lineage columns: {missing_gold_meta_columns}"
        )

    frame_truth_hash = extract_truth_hash(frame_value)
    if frame_truth_hash is None:
        raise ValueError(f"{frame_name} does not contain a readable meta__truth_hash value.")

    if frame_truth_hash != GOLD_TRUTH_HASH:
        raise ValueError(
            f"{frame_name} truth hash does not match GOLD_TRUTH_HASH:\n"
            f"dataframe={frame_truth_hash}\n"
            f"record={GOLD_TRUTH_HASH}"
        )

    frame_parent_values = frame_value["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
    if not frame_parent_values:
        raise ValueError(f"{frame_name} is missing populated meta__parent_truth_hash values.")

    if len(frame_parent_values) != 1:
        raise ValueError(f"{frame_name} has multiple parent truth hashes: {frame_parent_values}")

    if frame_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
        raise ValueError(
            f"{frame_name} parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
            f"dataframe_parent={frame_parent_values[0]}\n"
            f"gold_parent_truth={GOLD_PARENT_TRUTH_HASH}"
        )

if not Path(gold_truth_path).exists():
    raise FileNotFoundError(f"Gold truth file was not created: {gold_truth_path}")

loaded_gold_truth_raw = load_json(gold_truth_path)

loaded_gold_truth = require_dict(
    loaded_gold_truth_raw,
    "loaded_gold_truth",
)

loaded_gold_truth_hash = str(
    loaded_gold_truth.get("truth_hash", "")
).strip()

if loaded_gold_truth_hash != GOLD_TRUTH_HASH:
    raise ValueError(
        "Saved Gold truth file hash does not match GOLD_TRUTH_HASH:\n"
        f"file={loaded_gold_truth_hash}\n"
        f"record={GOLD_TRUTH_HASH}"
    )

loaded_gold_parent_truth_hash = str(
    loaded_gold_truth.get("parent_truth_hash", "")
).strip()

if loaded_gold_parent_truth_hash != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Gold truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_gold_parent_truth_hash}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

print("Gold PreProcessing lineage sanity check passed.")

----

# Gold Preprocessing SQL Write Cell

Target:
- gold.preprocessed_features

Purpose:
- Persist Gold preprocessed features into Postgres for downstream inspection, scoring traceability, and split tracking.


In [ ]:

WRITE_TO_POSTGRES = False

if WRITE_TO_POSTGRES:

    gold_preprocessing_sql_summary_dataframe = write_gold_preprocessed_features_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        notebook_globals=globals(),
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
        feature_set_id=globals().get("FEATURE_SET_ID", "gold_preprocessed_v1"),
    )

    display(gold_preprocessing_sql_summary_dataframe)
else:
    print("Postgres write skipped.")